# Water Quality Data Preprocessing and Feature Engineering


# Setup and Library Imports


In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd

ROOT_DIR = "."
for candidate in [".", "..", "../.."]:
    if os.path.exists(os.path.join(candidate, "src", "preprocessing.py")):
        ROOT_DIR = os.path.abspath(candidate)
        break

if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

from src.preprocessing import (
    run_cleaning_pipeline,
    engineer_features,
    split_data,
    impute_missing_values,
    scale_features,
    prepare_training_datasets,
    MODEL_INPUT_FEATURES,
    TARGET_REGRESSION,
    TARGET_CLASSIFICATION,
    TARGET_CLASSIFICATION_ORDINAL,
)

print("Project Root:", ROOT_DIR)
print("Total Model Features:", len(MODEL_INPUT_FEATURES))


Project Root: /Users/amansoni/Documents/Development/Personal Projects/Semester_5_AML_DL
Total Model Features: 15


# Load and Clean Dataset


In [2]:
raw_path = os.path.join(ROOT_DIR, "data", "Dataset", "Parquet", "Combined Data", "Combined_dataset.parquet")
cleaned_path = os.path.join(ROOT_DIR, "data", "GeneratedOutput", "02.Preprocessing&FeatureSelection", "cleaned_water_quality.parquet")

if os.path.exists(cleaned_path):
    df_clean = pd.read_parquet(cleaned_path)
else:
    df_clean, summary = run_cleaning_pipeline(raw_path=raw_path, output_path=cleaned_path)

print("Cleaned Data Dimensions:", df_clean.shape)
df_clean[["country", "area", "waterbody_type", "date", "ph", "temperature_c", "do_mg_l", "ccme_values", "ccme_wqi"]].head()


Cleaned Data Dimensions: (4287890, 23)


,country,area,waterbody_type,date,ph,temperature_c,do_mg_l,ccme_values,ccme_wqi
0,Australia,Torrens River Adelaide Zoo Pedestrian Bridge,River,2014-12-01,NaN,24.7,4.15,93.102127,Good
1,Australia,Torrens River Adelaide Zoo Pedestrian Bridge,River,2015-02-02,NaN,22.4,12.07,100.000000,Excellent
2,Australia,Torrens River Adelaide Zoo Pedestrian Bridge,River,2015-03-02,NaN,17.0,10.59,100.000000,Excellent
3,Australia,Torrens River Adelaide Zoo Pedestrian Bridge,River,2015-04-13,7.4,17.0,3.42,100.000000,Excellent
4,Australia,Torrens River Adelaide Zoo Pedestrian Bridge,River,2015-01-15,7.4,23.2,3.91,92.814413,Good


# Feature Engineering


In [ ]:
df_feat, new_cols = engineer_features(df_clean)
print("Engineered Features:", new_cols)
df_feat[["np_ratio", "bod_do_ratio", "do_deficit", "month_sin", "month_cos", "ccme_wqi_code"]].describe().round(4)

2026-10-25 21:36:44,294 [INFO] Engineered 7 new features: ['np_ratio', 'bod_do_ratio', 'do_deficit', 'month', 'quarter', 'month_sin', 'month_cos']


Engineered Features: ['np_ratio', 'bod_do_ratio', 'do_deficit', 'month', 'quarter', 'month_sin', 'month_cos']


,np_ratio,bod_do_ratio,do_deficit,month_sin,month_cos,ccme_wqi_code
count,2.817393e+06,2.821499e+06,3.140827e+06,4.287890e+06,4.287890e+06,4287890.0
mean,6.016730e+01,1.918900e+01,9.480000e-01,-3.500000e-03,-3.000000e-02,3.1843
std,3.630478e+02,3.953171e+03,1.941900e+00,7.105000e-01,7.031000e-01,1.0555
min,0.000000e+00,0.000000e+00,-1.691930e+01,-1.000000e+00,-1.000000e+00,0.0
25%,1.000000e+01,1.569000e-01,-3.750000e-02,-8.660000e-01,-8.660000e-01,3.0
50%,2.613640e+01,2.348000e-01,6.833000e-01,0.000000e+00,-0.000000e+00,4.0
75%,4.980390e+01,3.167000e-01,1.528600e+00,8.660000e-01,5.000000e-01,4.0
max,1.490000e+05,4.210000e+06,1.465200e+01,1.000000e+00,1.000000e+00,4.0


# Feature Selection and Target Quarantine


In [4]:
print("Model Input Features:")
for i, f in enumerate(MODEL_INPUT_FEATURES, 1):
    print(f"{i:2d}. {f}")

for target_col in [TARGET_REGRESSION, TARGET_CLASSIFICATION, TARGET_CLASSIFICATION_ORDINAL]:
    assert target_col not in MODEL_INPUT_FEATURES
print("Target quarantine check passed.")


Model Input Features:
 1. ph
 2. temperature_c
 3. do_mg_l
 4. bod_mg_l
 5. ammonia_mg_l
 6. orthophosphate_mg_l
 7. nitrogen_mg_l
 8. nitrate_mg_l
 9. np_ratio
10. bod_do_ratio
11. do_deficit
12. month
13. quarter
14. month_sin
15. month_cos
Target quarantine check passed.


# Split Dataset into Train, Validation, and Test Sets


In [5]:
train_raw, val_raw, test_raw = split_data(
    df_feat,
    train_size=0.70,
    val_size=0.15,
    test_size=0.15,
    method="stratified_random",
    random_state=42,
)
print("Train Rows:", f"{len(train_raw):,}")
print("Val Rows:  ", f"{len(val_raw):,}")
print("Test Rows: ", f"{len(test_raw):,}")


2026-10-25 21:37:03,164 [INFO] Split data (stratified_random): Train=3,001,523 (70.0%), Val=643,183 (15.0%), Test=643,184 (15.0%)


Train Rows: 3,001,523
Val Rows:   643,183
Test Rows:  643,184


# Impute Missing Values Using Training Set Medians


In [6]:
train_imp, val_imp, test_imp, imputer_info = impute_missing_values(
    train_raw, val_raw, test_raw, feature_cols=MODEL_INPUT_FEATURES
)
print("Remaining Missing Values in Modeling Features:")
print("Train:", int(train_imp[MODEL_INPUT_FEATURES].isna().sum().sum()))
print("Val:  ", int(val_imp[MODEL_INPUT_FEATURES].isna().sum().sum()))
print("Test: ", int(test_imp[MODEL_INPUT_FEATURES].isna().sum().sum()))


Remaining Missing Values in Modeling Features:
Train: 0
Val:   0
Test:  0


# Scale Modeling Features Using Training Statistics


In [7]:
train_scaled, val_scaled, test_scaled, scaler = scale_features(
    train_imp, val_imp, test_imp, feature_cols=MODEL_INPUT_FEATURES
)
train_means = train_scaled[MODEL_INPUT_FEATURES].mean().abs()
train_stds = train_scaled[MODEL_INPUT_FEATURES].std(ddof=0)
print("Maximum mean deviation from 0.0:", f"{train_means.max():.6e}")
print("Maximum std deviation from 1.0: ", f"{abs(train_stds - 1.0).max():.6e}")


Maximum mean deviation from 0.0: 1.316204e-15
Maximum std deviation from 1.0:  2.220446e-16


# Validate Data Quality and Schema Parity


In [8]:
assert list(train_imp[MODEL_INPUT_FEATURES].columns) == MODEL_INPUT_FEATURES
assert list(val_imp[MODEL_INPUT_FEATURES].columns) == MODEL_INPUT_FEATURES
assert list(test_imp[MODEL_INPUT_FEATURES].columns) == MODEL_INPUT_FEATURES
assert list(train_scaled[MODEL_INPUT_FEATURES].columns) == MODEL_INPUT_FEATURES

assert train_imp["ccme_values"].notna().all()
assert val_imp["ccme_values"].notna().all()
assert test_imp["ccme_values"].notna().all()
assert train_imp["ccme_wqi_code"].notna().all()
print("All data quality and schema validations passed successfully.")


All data quality and schema validations passed successfully.


# Verify Preprocessing Artifacts and Metadata


In [9]:
meta_path = os.path.join(ROOT_DIR, "data", "GeneratedOutput", "02.Preprocessing&FeatureSelection", "preprocessing_metadata.json")
with open(meta_path) as f:
    meta = json.load(f)

orig_count = meta["original_row_count"]
clean_count = meta["cleaned_row_count"]
train_count = meta["split_counts"]["train"]
val_count = meta["split_counts"]["val"]
test_count = meta["split_counts"]["test"]

print("Metadata Version:    ", meta["metadata_version"])
print("Total Master Records:", f"{orig_count:,}")
print("Cleaned Records:     ", f"{clean_count:,}")
print("Train Records:       ", f"{train_count:,}")
print("Validation Records:  ", f"{val_count:,}")
print("Test Records:        ", f"{test_count:,}")
print("Input Feature Count: ", meta["input_feature_count"])


Metadata Version:     2.0.0
Total Master Records: 4,297,895
Cleaned Records:      4,287,890
Train Records:        3,001,523
Validation Records:   643,183
Test Records:         643,184
Input Feature Count:  15
